In [ ]:
pip install sentence-transformers umap-learn hdbscan scikit-learn datasets

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

base = '/content/drive/MyDrive/news_explorer'
os.makedirs(f'{base}/data', exist_ok=True)
os.makedirs(f'{base}/outputs', exist_ok=True)

print("mounted and folders ready")
print(f"working directory: {base}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
mounted and folders ready
working directory: /content/drive/MyDrive/news_explorer


In [ ]:
from datasets import load_dataset
import pandas as pd

base = '/content/drive/MyDrive/news_explorer'

dataset = load_dataset("cc_news", split="train", streaming=True)
first = next(iter(dataset))
print("available columns:", list(first.keys()))

available columns: ['title', 'text', 'domain', 'date', 'description', 'url', 'image_url']


In [ ]:
from datasets import load_dataset
import pandas as pd
base = '/content/drive/MyDrive/news_explorer'
dataset = load_dataset("cc_news", split="train", streaming=True)

rows = []
for article in dataset.take(20000):
    rows.append({
        "title":   article["title"],
        "content": article["text"],
        "date":    article["date"],
        "url":     article["url"],
        "source":  article["domain"],
    })

df_raw = pd.DataFrame(rows)
df_raw.to_csv(f'{base}/data/cc_news_raw.csv', index=False)
print(f"saved {len(df_raw)} articles to Drive")
print(df_raw[["title", "date", "source"]].head())

saved 20000 articles to Drive
                                               title                 date  \
0        Daughter Duo is Dancing in The Same Company  2017-12-11 20:19:05   
1  New York City Ballet Announces Interim Leaders...  2017-12-11 17:02:55   
2  Watch Pennsylvania Ballet & Boston Ballet Face...  2018-02-02 21:58:13   
3                                        dance shoes  2018-04-24 19:00:11   
4  Rebecca Krohn on Her Retirement from New York ...  2017-10-06 14:44:51   

                   source  
0  www.pointemagazine.com  
1  www.pointemagazine.com  
2  www.pointemagazine.com  
3  www.pointemagazine.com  
4  www.pointemagazine.com  


In [ ]:
import pandas as pd

base = '/content/drive/MyDrive/news_explorer'
df = pd.read_csv(f'{base}/data/cc_news_raw.csv')

# parse dates
df['date'] = pd.to_datetime(df['date'], errors='coerce')

# drop anything with no title or date
df = df.dropna(subset=['title', 'date'])

# drop duplicates
df = df.drop_duplicates(subset='title')

# remove whitespace from titles
df['title'] = df['title'].str.strip()

# drop very short titles
df = df[df['title'].str.len() > 20]

# filter to a single year so the temporal story is clean
df = df[(df['date'] >= '2018-01-01') & (df['date'] <= '2018-12-31')]

# sample down to 7000
df = df.sample(n=7000, random_state=42).reset_index(drop=True)

# add a numeric day-of-year column
df['day'] = df['date'].dt.dayofyear

df.to_csv(f'{base}/data/cc_news_clean.csv', index=False)

print(f"clean dataset: {len(df)} articles")
print(f"date range: {df['date'].min()} to {df['date'].max()}")
print(df[['title', 'date', 'source']].head())

clean dataset: 7000 articles
date range: 2018-01-02 20:00:00 to 2018-07-08 00:00:00
                                               title                date  \
0  Reputed Philadelphia mob boss faces NYC fraud ... 2018-02-03 06:00:00   
1  Midday Home Invasion Has Neighbors Shaken In J... 2018-02-03 17:15:09   
2  Belgium's high scorers take on Brazil's miserl... 2018-07-03 18:00:00   
3  Justice for Junior: 2 more arrests in innocent... 2018-07-04 17:20:39   
4  Strong health sign-ups under Obamacare encoura... 2018-01-29 07:15:00   

                  source  
0  www.taiwannews.com.tw  
1   newyork.cbslocal.com  
2  www.taiwannews.com.tw  
3           abc7news.com  
4  www.taiwannews.com.tw  


In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer

base = '/content/drive/MyDrive/news_explorer'

model = SentenceTransformer('all-MiniLM-L6-v2')

titles = df['title'].tolist()

embeddings = model.encode(
    titles,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True
)

np.save(f'{base}/data/embeddings.npy', embeddings)

print(f"embeddings shape- {embeddings.shape}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/110 [00:00<?, ?it/s]

embeddings shape- (7000, 384)


In [ ]:
import umap
import numpy as np
base = '/content/drive/MyDrive/news_explorer'

embeddings = np.load(f'{base}/data/embeddings.npy')

reducer = umap.UMAP(
    n_components=3,
    n_neighbors=20,
    min_dist=0.1,
    metric='cosine',
    random_state=42,
    verbose=True
)

coords_3d = reducer.fit_transform(embeddings)

np.save(f'{base}/data/coords_3d.npy', coords_3d)

print(f"reduction done, shape is {coords_3d.shape}")
print(f"x range {coords_3d[:,0].min():.2f} to {coords_3d[:,0].max():.2f}")
print(f"y range {coords_3d[:,1].min():.2f} to {coords_3d[:,1].max():.2f}")
print(f"z range {coords_3d[:,2].min():.2f} to {coords_3d[:,2].max():.2f}")

/usr/local/lib/python3.12/dist-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


UMAP(angular_rp_forest=True, metric='cosine', n_components=3, n_jobs=1, n_neighbors=20, random_state=42, verbose=True)
Thu Apr 16 21:06:49 2026 Construct fuzzy simplicial set
Thu Apr 16 21:06:49 2026 Finding Nearest Neighbors
Thu Apr 16 21:06:49 2026 Building RP forest with 9 trees
Thu Apr 16 21:06:53 2026 NN descent for 13 iterations
	 1  /  13
	 2  /  13
	 3  /  13
	 4  /  13
	 5  /  13
	Stopping threshold met -- exiting after 5 iterations
Thu Apr 16 21:07:05 2026 Finished Nearest Neighbor Search
Thu Apr 16 21:07:10 2026 Construct embedding


Epochs completed:   0%|            0/500 [00:00]

	completed  0  /  500 epochs
	completed  50  /  500 epochs
	completed  100  /  500 epochs
	completed  150  /  500 epochs
	completed  200  /  500 epochs
	completed  250  /  500 epochs
	completed  300  /  500 epochs
	completed  350  /  500 epochs
	completed  400  /  500 epochs
	completed  450  /  500 epochs
Thu Apr 16 21:07:20 2026 Finished embedding
reduction done, shape is (7000, 3)
x range 1.84 to 9.21
y range 9.47 to 15.59
z range -3.74 to 1.50


In [ ]:
import hdbscan
import numpy as np

base = '/content/drive/MyDrive/news_explorer'

coords_3d = np.load(f'{base}/data/coords_3d.npy')

clusterer = hdbscan.HDBSCAN(
    min_cluster_size=60,
    min_samples=10,
    metric='euclidean',
    cluster_selection_method='eom',
    gen_min_span_tree=True
)

cluster_labels = clusterer.fit_predict(coords_3d)

n_clusters = len(set(cluster_labels)) - (1 if -1 in cluster_labels else 0)
n_noise     = (cluster_labels == -1).sum()

np.save(f'{base}/data/cluster_labels.npy', cluster_labels)

print(f"clusters found: {n_clusters}")
print(f"noise points  {n_noise} ({100 * n_noise / len(cluster_labels):.1f}%)")

clusters found: 27
noise points  3525 (50.4%)


In [ ]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
base = '/content/drive/MyDrive/news_explorer'

cluster_labels = np.load(f'{base}/data/cluster_labels.npy')
df = pd.read_csv(f'{base}/data/cc_news_clean.csv')

df['cluster'] = cluster_labels

unique_clusters = sorted([c for c in set(cluster_labels) if c != -1])

# concat all titles in each cluster into one document
cluster_docs = []
for c in unique_clusters:
    titles = df[df['cluster'] == c]['title'].tolist()
    cluster_docs.append(' '.join(titles))

vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    stop_words='english',
    max_features=5000
)

tfidf_matrix = vectorizer.fit_transform(cluster_docs)
feature_names = vectorizer.get_feature_names_out()

# pick top 3 terms
cluster_labels_dict = {}
for i, c in enumerate(unique_clusters):
    row = tfidf_matrix[i].toarray().flatten()
    top_indices = row.argsort()[-3:][::-1]
    label = ' | '.join(feature_names[top_indices])
    cluster_labels_dict[c] = label
    print(f"cluster {c:02d} ({(df['cluster']==c).sum()} articles): {label}")

cluster 00 (72 articles): opioid | abortion | penalty
cluster 01 (84 articles): facebook | cambridge | analytica
cluster 02 (61 articles): korea | north korea | korean
cluster 03 (185 articles): sunderland | cup | world cup
cluster 04 (104 articles): sox | yankees | dl
cluster 05 (75 articles): trump | macron | melania
cluster 06 (361 articles): super bowl | nba | lakers
cluster 07 (75 articles): school | students | teacher
cluster 08 (65 articles): fishing | whale | parry sound
cluster 09 (68 articles): french open | french | federer
cluster 10 (95 articles): memo | fbi | trump
cluster 11 (133 articles): state | beats | basketball
cluster 12 (82 articles): capitals | knights | stanley cup
cluster 13 (101 articles): crash | driving | self driving
cluster 14 (73 articles): china | tariffs | news
cluster 15 (184 articles): israel | syria | egypt
cluster 16 (598 articles): news | police | suspect
cluster 17 (71 articles): vote | gop | election
cluster 18 (125 articles): news | rouge | bat

In [ ]:
import numpy as np
import pandas as pd
import json
base = '/content/drive/MyDrive/news_explorer'

coords_3d     = np.load(f'{base}/data/coords_3d.npy')
cluster_labels = np.load(f'{base}/data/cluster_labels.npy')
df            = pd.read_csv(f'{base}/data/cc_news_clean.csv')

df['cluster'] = cluster_labels

# build points array
points = []
for i, row in df.iterrows():
    points.append({
        "x":       round(float(coords_3d[i, 0]), 4),
        "y":       round(float(coords_3d[i, 1]), 4),
        "z":       round(float(coords_3d[i, 2]), 4),
        "cluster": int(row['cluster']),
        "title":   row['title'],
        "date":    str(row['date']),
        "url":     row['url'],
        "source":  row['source'],
        "day":     int(row['day']),
    })

# build clusters array using labels 8
clusters = {
    -1:  "noise",
    0:   "opioid | abortion | penalty",
    1:   "facebook | cambridge | analytica",
    2:   "korea | north korea | korean",
    3:   "sunderland | cup | world cup",
    4:   "sox | yankees | dl",
    5:   "trump | macron | melania",
    6:   "super bowl | nba | lakers",
    7:   "school | students | teacher",
    8:   "fishing | whale | parry sound",
    9:   "french open | french | federer",
    10:  "memo | fbi | trump",
    11:  "state | beats | basketball",
    12:  "capitals | knights | stanley cup",
    13:  "crash | driving | self driving",
    14:  "china | tariffs | news",
    15:  "israel | syria | egypt",
    16:  "news | police | suspect",
    17:  "vote | gop | election",
    18:  "news | rouge | baton",
    19:  "news | energy | california",
    20:  "ceo | financial | banks",
    21:  "di | dan | yang",
    22:  "market | global | analysis",
    23:  "apple | iphone | microsoft",
    24:  "july | independence | day",
    25:  "dance | music | album",
    26:  "infinity | infinity war | netflix",
}

output = {
    "points":   points,
    "clusters": clusters,
}

with open(f'{base}/outputs/news_map.json', 'w') as f:
    json.dump(output, f)

size_mb = round(os.path.getsize(f'{base}/outputs/news_map.json') / 1e6, 2)
print(f"saved news_map.json - {len(points)} points, {size_mb} MB")

saved news_map.json - 7000 points, 2.09 MB


In [ ]:
# higher is better, range is -1 to 1
import numpy as np
coords_3d      = np.load('/content/drive/MyDrive/news_explorer/data/coords_3d.npy')
cluster_labels = np.load('/content/drive/MyDrive/news_explorer/data/cluster_labels.npy')

dbcv_score = clusterer.relative_validity_

print(f"DBCV score- {dbcv_score:.4f}")

DBCV score- 0.1994


In [ ]:
import numpy as np
from sklearn.manifold import trustworthiness
base = '/content/drive/MyDrive/news_explorer'

embeddings = np.load(f'{base}/data/embeddings.npy')
coords_3d  = np.load(f'{base}/data/coords_3d.npy')

# n_neighbors should match what is used in UMAP
score = trustworthiness(embeddings, coords_3d, n_neighbors=20)

print(f"trustworthiness score (UMAP 3D): {score:.4f}")

trustworthiness score (UMAP 3D): 0.9065


In [ ]:

import numpy as np
from sklearn.manifold import TSNE, trustworthiness
base = '/content/drive/MyDrive/news_explorer'

embeddings = np.load(f'{base}/data/embeddings.npy')

# t-SNE is slow  subsample 2000 points for a fair comparison
np.random.seed(42)
idx        = np.random.choice(len(embeddings), 2000, replace=False)
emb_sample = embeddings[idx]

tsne = TSNE(
    n_components=2,
    perplexity=30,
    n_iter=1000,
    random_state=42,
    verbose=1
)

coords_tsne = tsne.fit_transform(emb_sample)

score_tsne = trustworthiness(emb_sample, coords_tsne, n_neighbors=20)
print(f"trustworthiness score (t-SNE 2D): {score_tsne:.4f}")

# also recompute UMAP score on same subsample for fair comparison
from umap import UMAP
reducer_sub = UMAP(n_components=2, n_neighbors=20, random_state=42)
coords_umap_sub = reducer_sub.fit_transform(emb_sample)
score_umap = trustworthiness(emb_sample, coords_umap_sub, n_neighbors=20)
print(f"trustworthiness score (UMAP 2D):  {score_umap:.4f}")

/usr/local/lib/python3.12/dist-packages/sklearn/manifold/_t_sne.py:1164: FutureWarning: 'n_iter' was renamed to 'max_iter' in version 1.5 and will be removed in 1.7.
  warnings.warn(


[t-SNE] Computing 91 nearest neighbors...
[t-SNE] Indexed 2000 samples in 0.001s...
[t-SNE] Computed neighbors for 2000 samples in 0.314s...
[t-SNE] Computed conditional probabilities for sample 1000 / 2000
[t-SNE] Computed conditional probabilities for sample 2000 / 2000
[t-SNE] Mean sigma: 0.316286
[t-SNE] KL divergence after 50 iterations with early exaggeration: 78.256729
[t-SNE] KL divergence after 1000 iterations: 2.004015
trustworthiness score (t-SNE 2D): 0.8727


/usr/local/lib/python3.12/dist-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


trustworthiness score (UMAP 2D):  0.8511


In [ ]:

print("RESULTS SUMMARY")

print(f"dataset:          7000 articles, CC-News 2018")
print(f"embedding model:  all-MiniLM-L6-v2 (384 dims)")
print(f"clusters found:   27")
print(f"noise points:     3525 (50.4%)")
print(f"")
print(f"DBCV score:                    0.1994")
print(f"trustworthiness (UMAP 3D):     0.9065")
print(f"trustworthiness (t-SNE 2D):    0.8727")
print(f"trustworthiness (UMAP 2D):     0.8511")


RESULTS SUMMARY
dataset:          7000 articles, CC-News 2018
embedding model:  all-MiniLM-L6-v2 (384 dims)
clusters found:   27
noise points:     3525 (50.4%)

DBCV score:                    0.1994
trustworthiness (UMAP 3D):     0.9065
trustworthiness (t-SNE 2D):    0.8727
trustworthiness (UMAP 2D):     0.8511
